# NVDA Intrahour Volatility Dataset Construction

**Author:** Ayooluwa Adelagun

**Date:** March 2026

---

## Overview

This notebook constructs a supervised machine learning dataset for predicting the **direction of intrahour volatility** for NVIDIA Corporation (NVDA).

Intrahour volatility is estimated as the **Root Sum of Squares (RSS) of minute-level log returns** within each trading hour. The target variable is binary: `1` if the current hour's volatility is greater than the previous hour's, `0` otherwise.

### Data Sources
- `NVDA_full_1min_adjsplit.txt` — Split-adjusted 1-minute OHLCV data from 2000-01-03
- `NVDA_full_1hour_adjsplit.txt` — Split-adjusted hourly OHLCV data from 2000-01-03

### Pipeline Summary
1. Load 1-minute and hourly price data
2. Compute minute-level log returns and aggregate to hourly intrahour volatility
3. Remove hours with zero volatility
4. Align the volatility series with the hourly OHLCV data by matching timestamps
5. Engineer feature variables from the hourly price data
6. Construct the binary target variable
7. Export the final dataset to CSV

### Feature Variables
| Feature | Description |
|---|---|
| `Return_Squared` | Squared hourly log return (variance proxy) |
| `Hourly Volatility` | RSS of minute log returns (intrahour volatility estimate) |
| `target` | 1 if volatility increased vs prior hour, else 0 |

---
## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime

---
## 2. Load & Preprocess Data

Both files have no header row so column names are assigned manually.

**Minute log return** is computed as:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$$

where $P_t$ is the 1-minute closing price.

In [2]:
# Files have no header row so column names are assigned manually
data_1min = pd.read_csv('NVDA_full_1min_adjsplit.txt', header=None)
data_1min.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

# Convert Date to datetime for time-based resampling
data_1min['Date'] = pd.to_datetime(data_1min['Date'], format='%Y-%m-%d %H:%M:%S')

# Without a header row, pandas reads all columns as strings;
# Close must be cast to float before any numeric operations
data_1min['Close'] = data_1min['Close'].astype(float)

data_1hour = pd.read_csv('NVDA_full_1hour_adjsplit.txt', header=None)
data_1hour.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

# Minute log return: ln(P_t) - ln(P_{t-1})
# .diff() subtracts the previous row, giving the period-over-period log change
data_1min['Minute Log Return'] = np.log(data_1min['Close']).diff()

print(f"1-minute rows: {len(data_1min):,}")
print(f"Hourly rows:   {len(data_1hour):,}")
print(data_1min.head(10))

1-minute rows: 3,444,735
Hourly rows:   83,075
                 Date    Open    High     Low   Close   Volume  \
0 2000-01-03 09:30:00  0.0984  0.0990  0.0982  0.0990   768000   
1 2000-01-03 09:31:00  0.0984  0.0990  0.0983  0.0989  1968000   
2 2000-01-03 09:32:00  0.0984  0.0990  0.0984  0.0990   528000   
3 2000-01-03 09:33:00  0.0984  0.0990  0.0984  0.0990   288000   
4 2000-01-03 09:34:00  0.0987  0.0990  0.0984  0.0990   720000   
5 2000-01-03 09:35:00  0.0987  0.0987  0.0984  0.0987  1872000   
6 2000-01-03 09:36:00  0.0987  0.0987  0.0987  0.0987   240000   
7 2000-01-03 09:37:00  0.0987  0.0990  0.0987  0.0990   240000   
8 2000-01-03 09:38:00  0.0987  0.0990  0.0982  0.0990   192000   
9 2000-01-03 09:39:00  0.0982  0.0982  0.0982  0.0982   144000   

   Minute Log Return  
0                NaN  
1          -0.001011  
2           0.001011  
3           0.000000  
4           0.000000  
5          -0.003035  
6           0.000000  
7           0.003035  
8           0.00000

---
## 3. Compute Intrahour Volatility

Intrahour volatility for each trading hour is estimated as the **Root Sum of Squares (RSS)** of the minute-level log returns within that hour:

$$\sigma_{\text{intrahour}} = \sqrt{\sum_{i=1}^{N} r_i^2}$$

where $r_i$ are the minute log returns within the hour and $N$ is the number of minutes.

Minute data is resampled to hourly frequency using `.resample('h')`.

In [ ]:
def calculate_hourly_volatility(dataframe):
    # Work on a copy to avoid modifying the original dataframe
    dataframe = dataframe.copy()
    dataframe['Date'] = pd.to_datetime(dataframe['Date'], format='%Y-%m-%d %H:%M:%S')

    # Isolate the two columns needed; set Date as index for resampling
    date_log_return_df = dataframe[['Date', 'Minute Log Return']]
    date_log_return_df = date_log_return_df.set_index('Date')

    # Resample to hourly frequency and compute RSS of minute log returns:
    #   Step 1: square each return and sum within the hour (sum of squares)
    #   Step 2: take the square root -> RSS intrahour volatility estimate
    hourly_volatility = (
        date_log_return_df
        .resample('h')
        .apply(lambda x: (x ** 2).sum(axis=0))  # sum of squared returns
        .apply(lambda x: x ** 0.5)               # square root -> RSS
    )

    hourly_volatility.reset_index(inplace=True)
    hourly_volatility.columns = ['Date', 'Hourly_Volatility']
    return hourly_volatility

result_df = calculate_hourly_volatility(data_1min)
print(result_df.head(10))

                 Date  Hourly_Volatility
0 2000-01-03 09:00:00           0.021945
1 2000-01-03 10:00:00           0.038299
2 2000-01-03 11:00:00           0.033626
3 2000-01-03 12:00:00           0.022280
4 2000-01-03 13:00:00           0.013279
5 2000-01-03 14:00:00           0.034158
6 2000-01-03 15:00:00           0.024994
7 2000-01-03 16:00:00           0.000000
8 2000-01-03 17:00:00           0.000000
9 2000-01-03 18:00:00           0.000000


---
## 4. Remove Zero-Volatility Hours

Hours where the computed intrahour volatility is exactly zero are removed. These correspond to periods outside of active trading where no minute-level price movement occurred.

In [ ]:
def remove_zero_volatility(dataframe):
    # Exclude hours with zero volatility because these contain no valid minute return data
    filtered_dataframe = dataframe[dataframe['Hourly_Volatility'] != 0]
    return filtered_dataframe

newresults = remove_zero_volatility(result_df)
print(f"Rows before filtering: {len(result_df):,}")
print(f"Rows after filtering:  {len(newresults):,}")

Rows before filtering: 229,331
Rows after filtering:  82,026


---
## 5. Align Volatility with Hourly OHLCV Data

The intrahour volatility series (derived from minute data) and the hourly OHLCV data may not share identical timestamps. The minute data resamples to every hour on the hour, while the hourly file may have slightly different timestamps. An inner merge on the Date column keeps only rows present in both dataframes, ensuring perfect alignment before joining.

In [5]:
def matching_dataframes_bydate(newresults, data_1hour):
    # Convert Date to datetime in both dataframes to ensure consistent types for merging
    newresults = newresults.copy()
    data_1hour = data_1hour.copy()
    newresults['Date'] = pd.to_datetime(newresults['Date'])
    data_1hour['Date'] = pd.to_datetime(data_1hour['Date'], format='%Y-%m-%d %H:%M:%S')

    # Inner merge on Date keeps only timestamps present in both datasets
    common_dates = pd.merge(newresults[['Date']], data_1hour[['Date']], on='Date', how='inner')

    # Filter both dataframes to the common timestamps
    newresults_sync = newresults[newresults['Date'].isin(common_dates['Date'])]
    data_1hour_sync = data_1hour[data_1hour['Date'].isin(common_dates['Date'])]

    return newresults_sync, data_1hour_sync

newresults_sync, data_1hour_sync = matching_dataframes_bydate(newresults, data_1hour)
print(f"Aligned rows: {len(newresults_sync):,}")

Aligned rows: 82,026


---
## 6. Join Volatility onto Hourly Data

Both dataframes are set to a datetime index so pandas can align rows by timestamp when assigning the volatility column.

In [ ]:
# Set Date as index on both dataframes so pandas aligns rows by timestamp when joining
data_1hour_sync = data_1hour_sync.copy()
newresults_sync = newresults_sync.copy()

data_1hour_sync['Date'] = pd.to_datetime(data_1hour_sync['Date'])
newresults_sync['Date'] = pd.to_datetime(newresults_sync['Date'])
data_1hour_sync.set_index('Date', inplace=True)
newresults_sync.set_index('Date', inplace=True)

# Cast OHLCV columns to float. Read as strings due to missing header row
for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    data_1hour_sync[col] = data_1hour_sync[col].astype(float)

# Join intrahour volatility onto the hourly dataframe by matching datetime index
data_1hour_sync['Hourly Volatility'] = newresults_sync['Hourly_Volatility']

print(data_1hour_sync.head(10))

                       Open    High     Low   Close      Volume  \
Date                                                              
2000-01-03 09:00:00  0.0984  0.0990  0.0947  0.0957  28224000.0   
2000-01-03 10:00:00  0.0949  0.0958  0.0919  0.0928  53232000.0   
2000-01-03 11:00:00  0.0934  0.0947  0.0923  0.0947  22224000.0   
2000-01-03 12:00:00  0.0940  0.0964  0.0940  0.0949  29232000.0   
2000-01-03 13:00:00  0.0956  0.0958  0.0949  0.0951   6720000.0   
2000-01-03 14:00:00  0.0953  0.0979  0.0953  0.0966  31440000.0   
2000-01-03 15:00:00  0.0969  0.0978  0.0961  0.0975  80208000.0   
2000-01-04 09:00:00  0.0958  0.0961  0.0932  0.0939  46368000.0   
2000-01-04 10:00:00  0.0940  0.0952  0.0940  0.0952  68112000.0   
2000-01-04 11:00:00  0.0952  0.0961  0.0944  0.0951  18384000.0   

                     Hourly Volatility  
Date                                    
2000-01-03 09:00:00           0.021945  
2000-01-03 10:00:00           0.038299  
2000-01-03 11:00:00           0

---
## 7. Engineer Feature Variables

Two features are derived from the hourly closing price:

- **Return**: hourly log return, the log change in close price from one hour to the next
- **Return Squared**: squared hourly log return, a standard proxy for hourly variance

Rows with any missing values are dropped, which removes the first row (no prior close for log return).

In [7]:
# Hourly log return: ln(Close_t) - ln(Close_{t-1})
data_1hour_sync['Return'] = np.log(data_1hour_sync['Close']).diff()

# Squared return: proxy for hourly variance
data_1hour_sync['Return_Squared'] = np.square(data_1hour_sync['Return'])

# Drop NaNs: removes first row (no prior close) and any hours with missing volatility
data_1hour_sync = data_1hour_sync.dropna()

print(f"Rows after feature engineering: {len(data_1hour_sync):,}")
print(data_1hour_sync[['Close', 'Return', 'Return_Squared', 'Hourly Volatility']].head(10))

Rows after feature engineering: 82,025
                      Close    Return  Return_Squared  Hourly Volatility
Date                                                                    
2000-01-03 10:00:00  0.0928 -0.030772        0.000947           0.038299
2000-01-03 11:00:00  0.0947  0.020267        0.000411           0.033626
2000-01-03 12:00:00  0.0949  0.002110        0.000004           0.022280
2000-01-03 13:00:00  0.0951  0.002105        0.000004           0.013279
2000-01-03 14:00:00  0.0966  0.015650        0.000245           0.034158
2000-01-03 15:00:00  0.0975  0.009274        0.000086           0.024994
2000-01-04 09:00:00  0.0939 -0.037622        0.001415           0.031810
2000-01-04 10:00:00  0.0952  0.013750        0.000189           0.018919
2000-01-04 11:00:00  0.0951 -0.001051        0.000001           0.020631
2000-01-04 12:00:00  0.0943 -0.008448        0.000071           0.020995


---
## 8. Construct Target Variable

The target variable is a binary classification label indicating whether intrahour volatility **increased** relative to the prior hour:

$$\text{target}_t = \begin{cases} 1 & \text{if } \sigma_t > \sigma_{t-1} \\ 0 & \text{otherwise} \end{cases}$$

`.shift(1)` shifts the volatility series down by one row so each value is compared to the previous hour.

In [8]:
data = data_1hour_sync.dropna()
hourly_volatility = data['Hourly Volatility']

# Build a temporary dataframe to construct the target without risk of index misalignment
df = pd.DataFrame({'hourly_volatility': hourly_volatility})

# Target = 1 if this hour's volatility exceeds the previous hour's, else 0
# .shift(1) shifts the series down by one row so each value is compared to the prior hour
df['target'] = np.where(df['hourly_volatility'] > df['hourly_volatility'].shift(1), 1, 0)

# First row has no prior hour, so fill NaN with 0
df['target'] = df['target'].fillna(0)

# Join target back onto the main dataframe by datetime index
data['target'] = df['target']
data_full = data.dropna()

print(f"Final dataset shape: {data_full.shape}")
print(data_full['target'].value_counts())

Final dataset shape: (82025, 9)
target
0    45830
1    36195
Name: count, dtype: int64


---
## 9. Finalise Dataset

The dataset is trimmed to start from **2005-01-03** to remove the structurally different high-volatility regime observed in NVDA's early years (2000–2003).

In [ ]:
# Filter to start from 2005-01-03 and remove the structurally different high-volatility 2000-2003 regime
data_full = data_full[data_full.index >= '2005-01-03']

# Select columns in the canonical feature ordering for the modelling pipeline
data_full = data_full[['Return_Squared', 'Hourly Volatility', 'target']]

print(data_full.head(10))
print(f'\nDate range: {data_full.index.min()} to {data_full.index.max()}')
print(f'Total rows: {len(data_full):,}')

                     Return_Squared  Hourly Volatility  target
Date                                                          
2005-01-03 08:00:00        0.001264           0.037866       1
2005-01-03 09:00:00        0.000000           0.018079       0
2005-01-03 10:00:00        0.000337           0.015051       0
2005-01-03 11:00:00        0.000004           0.011035       0
2005-01-03 12:00:00        0.000001           0.005873       0
2005-01-03 13:00:00        0.000025           0.006292       1
2005-01-03 14:00:00        0.000001           0.007127       1
2005-01-03 15:00:00        0.000058           0.007494       1
2005-01-03 16:00:00        0.000016           0.010121       1
2005-01-03 17:00:00        0.000004           0.003208       0

Date range: 2005-01-03 08:00:00 to 2026-03-02 19:00:00
Total rows: 69,286


---
## 10. Export to CSV

The final dataset is exported to `NVDA_Intrahour_Volatility_Dataset.csv` with the datetime as the index.

In [10]:
# Export final dataset with datetime as the index
data_full.to_csv('NVDA_Intrahour_Volatility_Dataset.csv')
print('Saved to NVDA_Intrahour_Volatility_Dataset.csv')
print(f'Rows: {len(data_full):,}')
print(f'Date range: {data_full.index.min()} to {data_full.index.max()}')

Saved to NVDA_Intrahour_Volatility_Dataset.csv
Rows: 69,286
Date range: 2005-01-03 08:00:00 to 2026-03-02 19:00:00
